In [1]:
import os

In [2]:
%pwd

'c:\\Users\\mayur\\My_Study\\Python\\Chicken_Disease_Classification\\research'

In [3]:
# Cahnge directorry path
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\mayur\\My_Study\\Python\\Chicken_Disease_Classification'

In [5]:
# Update the entity: ie, return type of a function
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [6]:
from CNN_Classifier.constants import *
from CNN_Classifier.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
    ):
        self.config = read_yaml(config_filepath)   # to read config.yaml which has the artifacts of data ingestion  
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])  

    
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir,
        )

        return data_ingestion_config

In [8]:
import os
import urllib.request as request # to download file from URL
import zipfile   # to unzip the downloaded file
from CNN_Classifier import logger
from CNN_Classifier.utils.common import get_size # to get the size of file

In [15]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            os.makedirs(os.path.dirname(self.config.local_data_file), exist_ok=True)
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} download! with following info: {headers}")
        else:
            logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")

    def extract_zip_file(self):
        """file_path: str
        Extracts zip file into data dictionary"""
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok = True)
        with zipfile.ZipFile(self.config.local_data_file,'r') as zip_ref:
            zip_ref.extractall(unzip_path)
        logger.info(f"File extracted at : {unzip_path}")

In [17]:
# Create the pipeline
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-01-25 20:20:23,553: INFO: common]: yaml file: config\config.yaml loaded successfully
[2026-01-25 20:20:23,555: INFO: common]: yaml file: params.yaml loaded successfully
[2026-01-25 20:20:23,556: INFO: common]: Created directory at: artifacts
[2026-01-25 20:20:23,555: INFO: common]: yaml file: params.yaml loaded successfully
[2026-01-25 20:20:23,556: INFO: common]: Created directory at: artifacts
[2026-01-25 20:20:46,499: INFO: 1058941497]: artifacts/data_ingestion/data.zip download! with following info: Connection: close
Content-Length: 11616915
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "adf745abc03891fe493c3be264ec012691fe3fa21d861f35a27edbe6d86a76b1"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: A783:1BD4A3:15811AE:1B30A70:6976B2F5
Accept-Ranges: bytes
Date: Mon, 26 Jan 2026 